# 📏 Brouillon — Ridge Regression (L2)

> **Statut** : ⚠️ **Modèle écarté au profit de Lasso** — R² = 0.865, CV RMSE = 0.135  
> Ce notebook documente les expérimentations avec la régression Ridge (régularisation L2).  
> **Conclusion** : Performances légèrement inférieures à Lasso (L1), qui offre en plus la sélection automatique de features.

---
**Projet** : House Price Prediction — Ames Iowa  
**Auteur** : [Votre Nom]  
**Date** : Avril 2026

## 1. Imports & Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

from sklearn.linear_model import Ridge, Lasso, RidgeCV
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sys.path.append('../../src')
from preprocessing import full_pipeline

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train = pd.read_csv('../../data/train.csv')
test  = pd.read_csv('../../data/test.csv')
print(f'Train : {train.shape} | Test : {test.shape}')

## 2. Preprocessing

In [ ]:
X, X_test, y, scaler = full_pipeline(train, test)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Features : {X.shape[1]} | Train : {X_tr.shape[0]} | Val : {X_val.shape[0]}')

## 3. Ridge — Principe & Rappel Théorique

La régression Ridge minimise :
$$\text{RSS} + \lambda \sum_{j=1}^{p} \beta_j^2$$

- La pénalité **L2** réduit les coefficients vers 0 mais **ne les annule pas**
- Avantage : gère très bien la **multicolinéarité**
- Différence avec Lasso : Ridge conserve toutes les features (pas de sélection)

## 4. Essai 1 — Choix du paramètre alpha

In [ ]:
alphas = [0.01, 0.1, 0.5, 1, 5, 10, 20, 50, 100, 200, 500, 1000]
cv_rmse_vals, val_r2_vals = [], []

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    cv    = cross_val_score(ridge, X_tr, y_tr, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
    ridge.fit(X_tr, y_tr)
    cv_rmse_vals.append(-cv.mean())
    val_r2_vals.append(r2_score(y_val, ridge.predict(X_val)))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.semilogx(alphas, cv_rmse_vals, 'o-', color='#EF4444', label='CV RMSE')
ax1.set_xlabel('Alpha (log scale)')
ax1.set_ylabel('CV RMSE (log)', color='#EF4444')
ax1.set_title('Ridge — Sélection du paramètre alpha', fontweight='bold')

ax2 = ax1.twinx()
ax2.semilogx(alphas, val_r2_vals, 's--', color='#2563EB', label='Val R²')
ax2.set_ylabel('Val R²', color='#2563EB')

best_alpha_idx = val_r2_vals.index(max(val_r2_vals))
ax1.axvline(alphas[best_alpha_idx], color='gray', linestyle=':', alpha=.7)

plt.tight_layout()
plt.savefig('../../data/ridge_analysis.png', dpi=120)
plt.show()

print(f'Meilleur alpha : {alphas[best_alpha_idx]}')
print(f'CV RMSE       : {cv_rmse_vals[best_alpha_idx]:.4f}')
print(f'Val R²        : {val_r2_vals[best_alpha_idx]:.4f}')

## 5. Essai 2 — RidgeCV (validation croisée intégrée)

In [ ]:
alphas_range = np.logspace(-2, 3, 100)
ridge_cv = RidgeCV(alphas=alphas_range, cv=5, scoring='neg_root_mean_squared_error')
ridge_cv.fit(X_tr, y_tr)

print(f'Meilleur alpha (RidgeCV) : {ridge_cv.alpha_:.4f}')
print(f'Val R²                   : {r2_score(y_val, ridge_cv.predict(X_val)):.4f}')

## 6. Modèle final Ridge (alpha=10)

In [ ]:
ridge_best = Ridge(alpha=10)
cv_scores  = cross_val_score(ridge_best, X_tr, y_tr, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_best.fit(X_tr, y_tr)

preds    = ridge_best.predict(X_val)
cv_rmse  = -cv_scores.mean()
val_rmse = np.sqrt(mean_squared_error(y_val, preds))
val_r2   = r2_score(y_val, preds)
val_mae  = mean_absolute_error(y_val, preds)

print('=== Ridge (alpha=10) — Résultats finaux ===')
print(f'CV RMSE  : {cv_rmse:.4f}')
print(f'Val RMSE : {val_rmse:.4f}')
print(f'Val R²   : {val_r2:.4f}')
print(f'Val MAE  : {val_mae:.4f}')

## 7. Coefficients les plus importants

In [ ]:
coefs = pd.Series(np.abs(ridge_best.coef_), index=X.columns)
top15 = coefs.nlargest(15)

fig, ax = plt.subplots(figsize=(8, 5))
top15.plot.barh(ax=ax, color='#2563EB', alpha=.8)
ax.invert_yaxis()
ax.set_title('Top 15 Coefficients Ridge (|valeur|)', fontweight='bold')
ax.set_xlabel('|Coefficient|')
plt.tight_layout()
plt.show()

# Combien de coefficients ≈ 0 ?
near_zero = (np.abs(ridge_best.coef_) < 0.001).sum()
print(f'Coefficients ≈ 0 (|β| < 0.001) : {near_zero} / {len(ridge_best.coef_)}')
print('→ Ridge ne fait PAS de sélection de features (contrairement à Lasso)')

## 8. Comparaison Ridge vs Lasso

In [ ]:
lasso_best = Lasso(alpha=0.001, max_iter=5000)
cv_scores_lasso = cross_val_score(lasso_best, X_tr, y_tr, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_best.fit(X_tr, y_tr)

preds_l = lasso_best.predict(X_val)
zero_coefs = (lasso_best.coef_ == 0).sum()

print('=== Comparaison Ridge vs Lasso ===')
print(f'{'Métrique':<20} {'Ridge':>10} {'Lasso':>10}')
print('-' * 42)
print(f'{'CV RMSE':<20} {cv_rmse:>10.4f} {-cv_scores_lasso.mean():>10.4f}')
print(f'{'Val R²':<20} {val_r2:>10.4f} {r2_score(y_val, preds_l):>10.4f}')
print(f'{'Val MAE':<20} {val_mae:>10.4f} {mean_absolute_error(y_val, preds_l):>10.4f}')
print(f'{'Coefs nuls':<20} {near_zero:>10} {zero_coefs:>10}')
print()
print('→ Lasso sélectionne automatiquement', zero_coefs, 'features inutiles')
print('→ Lasso légèrement meilleur sur toutes les métriques → RETENU')

## 9. ⚠️ Conclusion — Pourquoi Ridge est écarté au profit de Lasso

| Métrique | Ridge | **Lasso** |
|----------|-------|----------|
| CV RMSE  | 0.135 | **0.130** |
| Val R²   | 0.865 | **0.871** |
| Sélection features | ❌ Non | ✅ Oui |
| Interprétabilité | Moyenne | **Meilleure** |

**Raisons d'élimination :**
1. **Performances légèrement inférieures** à Lasso sur CV RMSE et R²
2. **Pas de sélection de features** — Ridge conserve tous les coefficients (même inutiles)
3. **Moins interprétable** que Lasso pour un modèle linéaire
4. **Lasso est strictement meilleur** dans ce contexte avec 200+ features

> **Action** : ⚠️ Modèle écarté. Lasso est retenu comme représentant des modèles linéaires dans le notebook final.